In [0]:
%sql
WITH player_metrics AS (
    SELECT 
        CONCAT(Apellido, ', ', Nombre) AS Jugador,
        COUNT(CASE WHEN TiempoJuego > 0 THEN 1 END) as partidos_jugados,
        SUM(Puntos) / NULLIF(2 * (SUM(tiros_2_totales) + SUM(tiros_3_totales) + 0.44 * SUM(tiros_libres_totales)), 0) AS ts,
        (SUM(tiros_2_aciertos) + 1.5 * SUM(tiros_3_aciertos)) / NULLIF(SUM(tiros_2_totales) + SUM(tiros_3_totales), 0) AS efg,
        AVG(uso_pctg) AS uso_pctg
    FROM laliga.gold.fact_stats fs
    JOIN laliga.gold.dim_jugadores dj
    ON fs.IdJugador = dj.IdJugador
    WHERE fs.IdEquipo = '89192'
    GROUP BY fs.IdJugador, jugador
)
SELECT
    Jugador,
    partidos_jugados,
    ROUND(ts, 2) AS TS,
    ROUND(efg, 2) AS eFG,
    ROUND(uso_pctg, 2) AS USO
FROM player_metrics 
ORDER BY partidos_jugados DESC